In [6]:
import requests


response = requests.get("http://localhost:11434/api/tags")
print("Ollama is running!", response.json())

Ollama is running! {'models': [{'name': 'nomic-embed-text:latest', 'model': 'nomic-embed-text:latest', 'modified_at': '2026-06-12T07:11:10.311803471Z', 'size': 274302450, 'digest': '0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'nomic-bert', 'families': ['nomic-bert'], 'parameter_size': '137M', 'quantization_level': 'F16', 'context_length': 2048, 'embedding_length': 768}, 'capabilities': ['embedding']}, {'name': 'llama3.2:latest', 'model': 'llama3.2:latest', 'modified_at': '2026-06-12T07:10:23.087497531Z', 'size': 2019393189, 'digest': 'a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '3.2B', 'quantization_level': 'Q4_K_M', 'context_length': 131072, 'embedding_length': 3072}, 'capabilities': ['completion', 'tools']}]}


In [7]:
from langchain_ollama import OllamaLLM, OllamaEmbeddings

llm = OllamaLLM(base_url="http://localhost:11434", model="llama3.2")
embedding_model = OllamaEmbeddings(base_url="http://localhost:11434", model="nomic-embed-text")

print("LLM and embeddings ready!")

LLM and embeddings ready!


In [8]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3.2", base_url="http://localhost:11434")

response = llm.invoke("what is human brain in one line")
print(response)

The human brain is a complex, dynamic, and highly interconnected organ that processes information, controls bodily functions, and enables cognitive and emotional experiences, with approximately 100 billion neurons working together to form the intricate networks of its neural circuits.


RAG APPLICATION FOR OWN TEXT DATA


STEP1 PREPARE DOCUMENT FOR YOUR TEXT


In [9]:
from langchain_core.documents import Document
my_text="""Oldboy (Korean: 올드보이) is a 2003 South Korean action thriller film[4][5] directed and co-written by Park Chan-wook. A loose adaptation of the Japanese manga Old Boy by Garon Tsuchiya and Nobuaki Minegishi, the film follows Oh Dae-su (Choi Min-sik), who is imprisoned for 15 years without knowing the identity of his captor or his captor's motives. When he is released, Dae-su finds himself trapped in a web of conspiracy and violence as he seeks revenge against his captor and falls in love with a young sushi chef, Mi-do (Kang Hye-jung).

Oldboy attained critical acclaim and accolades worldwide, including winning the Grand Prix at the 2004 Cannes Film Festival, where it garnered high praise from Quentin Tarantino, the president of the jury. In the United States, film critic Roger Ebert stated that Oldboy is a "powerful film not because of what it depicts, but because of the depths of the human heart which it strips bare". The film's action sequences, particularly the single shot corridor fight sequence, also received commendation for their impressive execution.

The film's success led to two adaptations: an unauthorized Hindi remake in 2006 and an official American adaptation in 2013. As part of Park Chan-wook's The Vengeance Trilogy, it serves as the second installment, following Sympathy for Mr. Vengeance (2002) and preceding Lady Vengeance (2005)."""

docs = [Document(page_content=my_text, metadata={"source": "Movie", "documentID": "Doc1"})]
print(f"Loaded {len(docs)} document")

Loaded 1 document


STEP 2 : SPLITTING THE DOCUMENTS INTO CHUNKS

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f" Split into {len(chunks)} chunks ")
chunks

 Split into 5 chunks 


[Document(metadata={'source': 'Movie', 'documentID': 'Doc1'}, page_content="Oldboy (Korean: 올드보이) is a 2003 South Korean action thriller film[4][5] directed and co-written by Park Chan-wook. A loose adaptation of the Japanese manga Old Boy by Garon Tsuchiya and Nobuaki Minegishi, the film follows Oh Dae-su (Choi Min-sik), who is imprisoned for 15 years without knowing the identity of his captor or his captor's motives. When he is released, Dae-su finds himself trapped in a web of conspiracy and violence as he seeks revenge against his captor and falls in love with a"),
 Document(metadata={'source': 'Movie', 'documentID': 'Doc1'}, page_content='against his captor and falls in love with a young sushi chef, Mi-do (Kang Hye-jung).'),
 Document(metadata={'source': 'Movie', 'documentID': 'Doc1'}, page_content='Oldboy attained critical acclaim and accolades worldwide, including winning the Grand Prix at the 2004 Cannes Film Festival, where it garnered high praise from Quentin Tarantino, the p

STEP 3; CREATING EMBEDDINGS FOR THE CHUNKS


In [11]:
## TESTING
test_embedding = embedding_model.embed_query("Hello, this is a test.")
print(f"First 5 values: {test_embedding[:5]}")

First 5 values: [0.051712688, 0.026671084, -0.17385194, -0.032843705, 0.08032101]


In [12]:
from langchain_community.vectorstores import Chroma
vectorstore=Chroma.from_documents(documents=chunks,embedding=embedding_model)
print("Embeddings created and stored!")

Embeddings created and stored!


STEP 4 ; SEMANTIC SEARCH


In [13]:
vectorstore.similarity_search("Old Boy",k=3)

[Document(metadata={'source': 'Movie', 'documentID': 'Doc1'}, page_content="Oldboy (Korean: 올드보이) is a 2003 South Korean action thriller film[4][5] directed and co-written by Park Chan-wook. A loose adaptation of the Japanese manga Old Boy by Garon Tsuchiya and Nobuaki Minegishi, the film follows Oh Dae-su (Choi Min-sik), who is imprisoned for 15 years without knowing the identity of his captor or his captor's motives. When he is released, Dae-su finds himself trapped in a web of conspiracy and violence as he seeks revenge against his captor and falls in love with a"),
 Document(metadata={'documentID': 'Doc1', 'source': 'Movie'}, page_content='Oldboy attained critical acclaim and accolades worldwide, including winning the Grand Prix at the 2004 Cannes Film Festival, where it garnered high praise from Quentin Tarantino, the president of the jury. In the United States, film critic Roger Ebert stated that Oldboy is a "powerful film not because of what it depicts, but because of the depths

Talk to LLm

In [15]:
context=vectorstore.similarity_search("Old Boy",k=3)
response=llm.invoke("Story of old boy movie?:{context}")
print(response)

You're referring to the classic South Korean film "Oldboy"!

Released in 2003, "Oldboy" is a psychological thriller directed by Park Chan-wook and starring Choi Min-sik, Kim Tae-woo, and Ryu Seung-ryeong.

The story follows Oh Dae-su (played by Choi Min-sik), a man who wakes up after 15 years in prison with no memory of his past. He is released on the condition that he must find his own wife and kill her for their shared lover, but things take a dark turn when he discovers that his wife has been brutally murdered.

As Dae-su navigates the complexities of his situation, he begins to unravel the truth behind his imprisonment and the mysterious circumstances surrounding his release. Along the way, he encounters a series of bizarre events, including the appearance of his captor, Lee Woo-jin (played by Kim Tae-woo), who seems to be manipulating him for his own twisted purposes.

Throughout the film, Dae-su's quest for revenge and justice becomes increasingly entangled with his own identity 